# Advanced Python 3.10 Problems with Complete Solutions

## Structural Pattern Matching and Strict `zip`

This notebook develops advanced, practical skill with two Python 3.10 additions:

1. Structural pattern matching with `match` / `case`
2. Length-safe parallel iteration with `zip(..., strict=True)`

Every problem includes:

- a precise specification;
- edge cases and constraints;
- a complete reference solution;
- executable tests;
- best-practice notes;
- optional extensions with solutions.

The notebook uses only the Python standard library and is designed for Python **3.10 or later**.

## Learning objectives

By the end, you should be able to:

- choose literal, OR, capture, wildcard, sequence, mapping, and class patterns;
- use guards without hiding essential validation logic;
- order cases from specific to general;
- avoid accidental capture patterns and unreachable cases;
- model recursive data with class patterns;
- validate synchronized iterables without consuming them twice;
- use strict zipping in data pipelines, matrix operations, and streaming code;
- combine pattern matching and strict zipping in maintainable production-style code.

In [1]:
import sys

assert sys.version_info >= (3, 10), "This notebook requires Python 3.10 or newer."
print("Python version:", sys.version.split()[0])

Python version: 3.13.7


## Best-practice checklist

### Pattern matching

- Put **specific cases before general cases**.
- Use `_` for a true wildcard.
- Remember that an unqualified name in a pattern is normally a **capture**, not a constant comparison.
- Keep guards short, deterministic, and free of side effects.
- Use mapping patterns for structured dictionaries rather than long chains of indexing.
- Use class patterns for domain models with stable, explicit structure.
- Raise informative exceptions at system boundaries.
- Do not force `match` into code where a dictionary lookup or polymorphism is clearer.

### Strict zipping

- Use `strict=True` whenever equal lengths are part of the data contract.
- Do not convert iterators to lists merely to compare lengths before zipping.
- Treat a strict-zip `ValueError` as a data-integrity failure.
- Add context by catching and re-raising with a domain-specific message.
- Remember that an iterator may be partially consumed before a mismatch is discovered.

# Fast reference examples

## Example A — literals, OR patterns, captures, guards, and wildcard

In [2]:
def classify_http_status(status: int) -> str:
    match status:
        case 200 | 201 | 204:
            return "success"
        case code if 300 <= code < 400:
            return "redirect"
        case code if 400 <= code < 500:
            return "client error"
        case code if 500 <= code < 600:
            return "server error"
        case _:
            return "non-standard"

assert classify_http_status(201) == "success"
assert classify_http_status(404) == "client error"
assert classify_http_status(503) == "server error"
assert classify_http_status(799) == "non-standard"

## Example B — sequence patterns and starred captures

A sequence pattern can decompose lists and tuples. Strings and bytes are deliberately not treated as sequence-pattern subjects.

In [3]:
def parse_path(parts):
    match parts:
        case ["users", int(user_id)]:
            return ("user", user_id)
        case ["users", int(user_id), "posts", *post_path]:
            return ("user-posts", user_id, post_path)
        case ["health"]:
            return ("health",)
        case _:
            raise ValueError(f"Unsupported path: {parts!r}")

assert parse_path(["users", 42]) == ("user", 42)
assert parse_path(["users", 42, "posts", "2026", "welcome"]) == (
    "user-posts", 42, ["2026", "welcome"]
)

## Example C — mapping patterns and `**rest`

Mapping patterns match required keys and ignore additional keys unless you capture them.

In [4]:
def extract_metric(message: dict) -> tuple[str, float, dict]:
    match message:
        case {
            "kind": "metric",
            "name": str(name),
            "value": int(value) | float(value),
            **rest,
        }:
            return name, float(value), rest
        case _:
            raise ValueError("Not a valid metric message")

name, value, extra = extract_metric(
    {"kind": "metric", "name": "latency_ms", "value": 12, "host": "api-1"}
)
assert (name, value) == ("latency_ms", 12.0)
assert extra == {"host": "api-1"}

## Example D — class patterns

In [5]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: float
    y: float

def point_region(point: Point) -> str:
    match point:
        case Point(0, 0):
            return "origin"
        case Point(x, 0):
            return f"x-axis at {x}"
        case Point(0, y):
            return f"y-axis at {y}"
        case Point(x, y) if x == y:
            return "diagonal x=y"
        case Point(x, y):
            return f"general point ({x}, {y})"

assert point_region(Point(0, 0)) == "origin"
assert point_region(Point(3, 0)) == "x-axis at 3"
assert point_region(Point(2, 2)) == "diagonal x=y"

## Example E — strict `zip`

In [6]:
names = iter(["Ada", "Grace", "Linus"])
scores = iter([98, 95, 91])

paired = list(zip(names, scores, strict=True))
assert paired == [("Ada", 98), ("Grace", 95), ("Linus", 91)]

try:
    list(zip([1, 2, 3], ["a", "b"], strict=True))
except ValueError as exc:
    print("Expected mismatch:", exc)
else:
    raise AssertionError("A length mismatch should have raised ValueError")

Expected mismatch: zip() argument 2 is shorter than argument 1


# Problem 1 — Production event normalizer

You receive event dictionaries from several producers. Normalize supported events into immutable domain objects.

## Requirements

Support:

- `user.created` with a positive integer `id`, a non-empty `name`, and a syntactically simple email check;
- `user.deleted` with a positive integer `id` and optional reason;
- `invoice.paid` with a positive integer `invoice_id`, a three-letter uppercase currency, and a positive numeric amount;
- reject unsupported or malformed events with an informative `ValueError`;
- retain unrecognized payload fields as metadata.

Use nested mapping patterns, class patterns for primitive types, captures, and guards.

In [7]:
# Sample inputs for Problem 1

EVENTS = [
    {
        "type": "user.created",
        "payload": {
            "id": 101,
            "name": "Ada",
            "email": "ada@example.com",
            "source": "referral",
        },
    },
    {
        "type": "user.deleted",
        "payload": {"id": 102, "reason": "duplicate"},
    },
    {
        "type": "invoice.paid",
        "payload": {
            "invoice_id": 9001,
            "currency": "EUR",
            "amount": 125.50,
            "processor": "demo-pay",
        },
    },
]

## Solution 1

In [8]:
from dataclasses import dataclass, field
from typing import Any

@dataclass(frozen=True)
class UserCreated:
    user_id: int
    name: str
    email: str
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class UserDeleted:
    user_id: int
    reason: str | None
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class InvoicePaid:
    invoice_id: int
    currency: str
    amount: float
    metadata: dict[str, Any] = field(default_factory=dict)


def normalize_event(event: object) -> UserCreated | UserDeleted | InvoicePaid:
    match event:
        case {
            "type": "user.created",
            "payload": {
                "id": int(user_id),
                "name": str(name),
                "email": str(email),
                **metadata,
            },
        } if (
            type(user_id) is int
            and user_id > 0
            and bool(name.strip())
            and "@" in email
            and not email.startswith("@")
            and not email.endswith("@")
        ):
            return UserCreated(
                user_id=user_id,
                name=name.strip(),
                email=email.strip(),
                metadata=metadata,
            )

        case {
            "type": "user.deleted",
            "payload": {
                "id": int(user_id),
                "reason": str(reason),
                **metadata,
            },
        } if type(user_id) is int and user_id > 0:
            return UserDeleted(user_id, reason.strip() or None, metadata)

        case {
            "type": "user.deleted",
            "payload": {"id": int(user_id), **metadata},
        } if type(user_id) is int and user_id > 0:
            return UserDeleted(user_id, None, metadata)

        case {
            "type": "invoice.paid",
            "payload": {
                "invoice_id": int(invoice_id),
                "currency": str(currency),
                "amount": int(amount) | float(amount),
                **metadata,
            },
        } if (
            type(invoice_id) is int
            and invoice_id > 0
            and len(currency) == 3
            and currency.isalpha()
            and currency.isupper()
            and type(amount) in (int, float)
            and amount > 0
        ):
            return InvoicePaid(invoice_id, currency, float(amount), metadata)

        case {"type": str(event_type)}:
            raise ValueError(f"Malformed or unsupported event type: {event_type!r}")

        case _:
            raise ValueError(f"Event must be a mapping with a string 'type': {event!r}")

In [9]:
normalized = [normalize_event(event) for event in EVENTS]

assert normalized[0] == UserCreated(
    101, "Ada", "ada@example.com", {"source": "referral"}
)
assert normalized[1] == UserDeleted(102, "duplicate", {})
assert normalized[2] == InvoicePaid(
    9001, "EUR", 125.5, {"processor": "demo-pay"}
)

invalid_events = [
    {"type": "user.created", "payload": {"id": -1, "name": "X", "email": "x@y"}},
    {"type": "invoice.paid", "payload": {"invoice_id": 1, "currency": "eur", "amount": 5}},
    {"payload": {}},
]

for bad_event in invalid_events:
    try:
        normalize_event(bad_event)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected rejection: {bad_event!r}")

normalized

[UserCreated(user_id=101, name='Ada', email='ada@example.com', metadata={'source': 'referral'}),
 UserDeleted(user_id=102, reason='duplicate', metadata={}),
 InvoicePaid(invoice_id=9001, currency='EUR', amount=125.5, metadata={'processor': 'demo-pay'})]

### Best-practice notes

- The fallback `case {"type": str(event_type)}` preserves context in the error.
- `type(value) is int` excludes booleans, which otherwise satisfy `isinstance(True, int)`.
- Mapping patterns naturally express required fields.
- The guards contain validation predicates, while construction remains in the case body.

# Problem 2 — Recursive expression evaluator and simplifier

Build a tiny arithmetic abstract syntax tree.

## Requirements

Implement:

- numeric literals;
- addition, subtraction, multiplication, division, and unary negation;
- evaluation;
- algebraic simplification rules such as `x + 0 -> x`, `x * 1 -> x`, and constant folding;
- a clear exception for division by zero;
- recursive class patterns.

## Solution 2

In [10]:
from dataclasses import dataclass
from typing import TypeAlias

@dataclass(frozen=True)
class Num:
    value: float

@dataclass(frozen=True)
class Add:
    left: "Expr"
    right: "Expr"

@dataclass(frozen=True)
class Sub:
    left: "Expr"
    right: "Expr"

@dataclass(frozen=True)
class Mul:
    left: "Expr"
    right: "Expr"

@dataclass(frozen=True)
class Div:
    left: "Expr"
    right: "Expr"

@dataclass(frozen=True)
class Neg:
    operand: "Expr"

Expr: TypeAlias = Num | Add | Sub | Mul | Div | Neg

In [11]:
def evaluate(expr: Expr) -> float:
    match expr:
        case Num(value):
            return float(value)
        case Add(left, right):
            return evaluate(left) + evaluate(right)
        case Sub(left, right):
            return evaluate(left) - evaluate(right)
        case Mul(left, right):
            return evaluate(left) * evaluate(right)
        case Div(left, right):
            denominator = evaluate(right)
            if denominator == 0:
                raise ZeroDivisionError("Division by zero in expression tree")
            return evaluate(left) / denominator
        case Neg(operand):
            return -evaluate(operand)
        case _:
            raise TypeError(f"Unknown expression node: {expr!r}")


def simplify(expr: Expr) -> Expr:
    match expr:
        # Recursively simplify children first.
        case Add(left, right):
            left_s = simplify(left)
            right_s = simplify(right)
            match (left_s, right_s):
                case (Num(0), other) | (other, Num(0)):
                    return other
                case (Num(a), Num(b)):
                    return Num(a + b)
                case _:
                    return Add(left_s, right_s)

        case Sub(left, right):
            left_s = simplify(left)
            right_s = simplify(right)
            match (left_s, right_s):
                case (other, Num(0)):
                    return other
                case (Num(a), Num(b)):
                    return Num(a - b)
                case _:
                    return Sub(left_s, right_s)

        case Mul(left, right):
            left_s = simplify(left)
            right_s = simplify(right)
            match (left_s, right_s):
                case (Num(0), _) | (_, Num(0)):
                    return Num(0)
                case (Num(1), other) | (other, Num(1)):
                    return other
                case (Num(a), Num(b)):
                    return Num(a * b)
                case _:
                    return Mul(left_s, right_s)

        case Div(left, right):
            left_s = simplify(left)
            right_s = simplify(right)
            match (left_s, right_s):
                case (_, Num(0)):
                    raise ZeroDivisionError("Division by zero during simplification")
                case (other, Num(1)):
                    return other
                case (Num(a), Num(b)):
                    return Num(a / b)
                case _:
                    return Div(left_s, right_s)

        case Neg(operand):
            operand_s = simplify(operand)
            match operand_s:
                case Num(value):
                    return Num(-value)
                case Neg(inner):
                    return inner
                case _:
                    return Neg(operand_s)

        case Num():
            return expr

        case _:
            raise TypeError(f"Unknown expression node: {expr!r}")

In [12]:
expression = Div(
    Mul(
        Add(Num(2), Num(0)),
        Add(Num(3), Num(4)),
    ),
    Num(1),
)

simplified = simplify(expression)

assert evaluate(expression) == 14
assert simplified == Num(14)
assert evaluate(simplified) == 14
assert simplify(Neg(Neg(Num(9)))) == Num(9)

try:
    evaluate(Div(Num(1), Num(0)))
except ZeroDivisionError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Expected ZeroDivisionError")

simplified

Expected: Division by zero in expression tree


Num(value=14)

### Why this design works

Class patterns make recursive domain models readable. The code also illustrates an important rule for OR patterns: every alternative must bind the same names. For example, `(Num(0), other) | (other, Num(0))` binds `other` in both alternatives.

# Problem 3 — Recursive robot command language

Design and execute a compact robot language.

## Command forms

- `["move", DIRECTION, STEPS]`
- `["pick", ITEM]`
- `["drop", ITEM]`
- `["repeat", COUNT, COMMAND_1, COMMAND_2, ...]`

Directions are `N`, `S`, `E`, and `W`. Counts and steps must be positive integers. The robot tracks position and inventory.

Use recursive sequence patterns, captures, OR patterns, guards, and informative errors.

## Solution 3

In [13]:
from dataclasses import dataclass, field

@dataclass
class RobotState:
    x: int = 0
    y: int = 0
    inventory: list[str] = field(default_factory=list)
    log: list[str] = field(default_factory=list)


def execute_command(command: object, state: RobotState) -> None:
    match command:
        case ["move", ("N" | "S" | "E" | "W") as direction, int(steps)] if (
            type(steps) is int and steps > 0
        ):
            dx, dy = {
                "N": (0, 1),
                "S": (0, -1),
                "E": (1, 0),
                "W": (-1, 0),
            }[direction]
            state.x += dx * steps
            state.y += dy * steps
            state.log.append(f"move {direction} {steps}")

        case ["pick", str(item)] if item.strip():
            item = item.strip()
            state.inventory.append(item)
            state.log.append(f"pick {item}")

        case ["drop", str(item)] if item.strip():
            item = item.strip()
            try:
                state.inventory.remove(item)
            except ValueError as exc:
                raise ValueError(f"Cannot drop absent item: {item!r}") from exc
            state.log.append(f"drop {item}")

        case ["repeat", int(count), *commands] if (
            type(count) is int and count > 0 and commands
        ):
            for _ in range(count):
                for nested_command in commands:
                    execute_command(nested_command, state)

        case ["move", *_]:
            raise ValueError(f"Malformed move command: {command!r}")

        case ["repeat", *_]:
            raise ValueError(f"Malformed repeat command: {command!r}")

        case _:
            raise ValueError(f"Unknown command: {command!r}")


def run_program(program: list[object]) -> RobotState:
    state = RobotState()
    for command in program:
        execute_command(command, state)
    return state

In [14]:
program = [
    ["pick", "sample"],
    [
        "repeat",
        2,
        ["move", "E", 3],
        ["move", "N", 1],
    ],
    ["drop", "sample"],
    ["move", "W", 2],
]

state = run_program(program)

assert (state.x, state.y) == (4, 2)
assert state.inventory == []
assert len(state.log) == 7
state

RobotState(x=4, y=2, inventory=[], log=['pick sample', 'move E 3', 'move N 1', 'move E 3', 'move N 1', 'drop sample', 'move W 2'])

### Extension — static validation before execution

A separate validator avoids partially modifying state when a later command is malformed.

In [15]:
def validate_command(command: object) -> None:
    match command:
        case ["move", ("N" | "S" | "E" | "W"), int(steps)] if (
            type(steps) is int and steps > 0
        ):
            return
        case [("pick" | "drop"), str(item)] if item.strip():
            return
        case ["repeat", int(count), *commands] if (
            type(count) is int and count > 0 and commands
        ):
            for nested in commands:
                validate_command(nested)
        case _:
            raise ValueError(f"Invalid command: {command!r}")


def run_validated_program(program: list[object]) -> RobotState:
    for command in program:
        validate_command(command)
    return run_program(program)


assert run_validated_program(program) == state

# Problem 4 — HTTP response decoder

Decode several response shapes from an HTTP client.

## Supported shapes

- successful JSON: `(status, headers, {"data": payload})`;
- paginated JSON: `(status, headers, {"data": items, "next": token})`;
- not found: status `404`;
- retryable server failures: `500`, `502`, `503`, or `504`, optionally with `Retry-After`;
- other client and server errors.

Headers are plain dictionaries. Use tuple, mapping, OR, capture, wildcard, and guard patterns.

## Solution 4

In [16]:
from dataclasses import dataclass
from typing import Any

@dataclass(frozen=True)
class Success:
    payload: Any

@dataclass(frozen=True)
class Page:
    items: list[Any]
    next_token: str | None

@dataclass(frozen=True)
class NotFound:
    message: str

@dataclass(frozen=True)
class RetryableError:
    status: int
    retry_after: int | None
    message: str

@dataclass(frozen=True)
class FatalError:
    status: int
    message: str


def decode_response(response: object):
    match response:
        case (
            int(status),
            dict(headers),
            {"data": list(items), "next": token},
        ) if 200 <= status < 300 and (token is None or isinstance(token, str)):
            return Page(items, token)

        case (int(status), dict(headers), {"data": payload}) if 200 <= status < 300:
            return Success(payload)

        case (404, _, {"error": str(message)}):
            return NotFound(message)

        case (
            (500 | 502 | 503 | 504) as status,
            {"Retry-After": str(raw_retry), **extra_fields},
            {"error": str(message)},
        ) if raw_retry.isdigit():
            return RetryableError(status, int(raw_retry), message)

        case (
            (500 | 502 | 503 | 504) as status,
            _,
            {"error": str(message)},
        ):
            return RetryableError(status, None, message)

        case (int(status), _, {"error": str(message)}) if 400 <= status < 600:
            return FatalError(status, message)

        case _:
            raise ValueError(f"Unrecognized response shape: {response!r}")

In [17]:
assert decode_response(
    (200, {"Content-Type": "application/json"}, {"data": {"id": 1}})
) == Success({"id": 1})

assert decode_response(
    (200, {}, {"data": [1, 2], "next": "page-2"})
) == Page([1, 2], "page-2")

assert decode_response(
    (503, {"Retry-After": "15"}, {"error": "busy"})
) == RetryableError(503, 15, "busy")

assert decode_response(
    (404, {}, {"error": "missing"})
) == NotFound("missing")

### Case-order lesson

The paginated-success case must appear before the general success case. Otherwise, the general `{"data": payload}` pattern would match first and pagination metadata would be lost.

# Problem 5 — State-machine transition engine

Implement a document workflow with states:

`draft -> review -> approved -> published`

Also allow:

- rejection from `review` back to `draft`;
- withdrawal from `review` or `approved` back to `draft`;
- publishing only when an approval timestamp exists;
- no transitions out of `published`.

Represent an action as a tuple `(current_state, action, context)`.

## Solution 5

In [18]:
from dataclasses import dataclass
from datetime import datetime, timezone

@dataclass(frozen=True)
class TransitionResult:
    old_state: str
    new_state: str
    audit_message: str


def transition(
    current_state: str,
    action: str,
    context: dict,
) -> TransitionResult:
    subject = (current_state, action, context)

    match subject:
        case ("draft", "submit", {"actor": str(actor), **extra_fields}):
            return TransitionResult("draft", "review", f"submitted by {actor}")

        case ("review", "approve", {"actor": str(actor), **extra_fields}):
            return TransitionResult("review", "approved", f"approved by {actor}")

        case ("review", "reject", {"actor": str(actor), "reason": str(reason), **extra_fields}) if reason.strip():
            return TransitionResult(
                "review", "draft", f"rejected by {actor}: {reason.strip()}"
            )

        case (
            "approved",
            "publish",
            {"actor": str(actor), "approved_at": datetime() as approved_at, **extra_fields},
        ) if approved_at.tzinfo is not None:
            return TransitionResult(
                "approved",
                "published",
                f"published by {actor}; approval recorded at {approved_at.isoformat()}",
            )

        case (("review" | "approved") as old_state, "withdraw", {"actor": str(actor), **extra_fields}):
            return TransitionResult(old_state, "draft", f"withdrawn by {actor}")

        case ("published", _, _):
            raise ValueError("Published documents are immutable")

        case _:
            raise ValueError(
                f"Illegal transition: state={current_state!r}, action={action!r}"
            )

In [19]:
approved_at = datetime(2026, 1, 15, tzinfo=timezone.utc)

assert transition("draft", "submit", {"actor": "Ada"}).new_state == "review"
assert transition("review", "approve", {"actor": "Grace"}).new_state == "approved"
assert transition(
    "approved",
    "publish",
    {"actor": "Linus", "approved_at": approved_at},
).new_state == "published"

try:
    transition("published", "withdraw", {"actor": "Ada"})
except ValueError as exc:
    assert "immutable" in str(exc)
else:
    raise AssertionError("Published state must reject transitions")

# Problem 6 — Diagnose pattern-matching bugs

The following intentions are common, but the implementations are wrong:

1. Compare against a constant named `RED`.
2. Distinguish `bool` from `int`.
3. Validate that every robot direction exists in the allowed set.
4. Prevent a general case from shadowing a specific one.

Write corrected implementations and explain each bug.

## Solution 6A — qualified constants instead of accidental captures

In [20]:
from enum import Enum

class Color(Enum):
    RED = "red"
    BLUE = "blue"


def color_message(color: object) -> str:
    match color:
        case Color.RED:
            return "stop"
        case Color.BLUE:
            return "continue"
        case _:
            return "unknown"


assert color_message(Color.RED) == "stop"
assert color_message("red") == "unknown"

An unqualified `case RED:` would be a capture pattern and would match almost anything. A qualified enum member such as `Color.RED` is a value pattern.

## Solution 6B — `bool` is a subclass of `int`

In [21]:
def numeric_kind(value: object) -> str:
    match value:
        case bool():
            return "boolean"
        case int():
            return "integer"
        case float():
            return "float"
        case _:
            return "other"


assert numeric_kind(True) == "boolean"
assert numeric_kind(1) == "integer"

Put the `bool()` case before `int()`. Otherwise, booleans are accepted by the broader integer class pattern.

## Solution 6C — subset versus proper subset

In [22]:
ALLOWED_DIRECTIONS = {"N", "S", "E", "W"}

def validate_directions(directions: list[str]) -> bool:
    # <= means "is a subset of"; < means "is a proper subset of".
    return set(directions) <= ALLOWED_DIRECTIONS


assert validate_directions(["N", "N", "W"])
assert validate_directions(["N", "S", "E", "W"])
assert not validate_directions(["UP"])

Using `<` would incorrectly reject the valid case where all four allowed directions occur, because the sets would be equal rather than one being a proper subset.

## Solution 6D — specific before general

In [23]:
def decode_message(message: dict) -> str:
    match message:
        case {"type": "error", "code": 404, **extra_fields}:
            return "not found"
        case {"type": "error", "code": int(code), **extra_fields}:
            return f"error {code}"
        case {"type": str(kind), **extra_fields}:
            return f"generic {kind}"
        case _:
            return "invalid"


assert decode_message({"type": "error", "code": 404}) == "not found"
assert decode_message({"type": "error", "code": 500}) == "error 500"

# Problem 7 — Strict records from headers and rows

Create `records_from_rows(headers, rows)`.

## Requirements

- each row must have exactly the same number of values as `headers`;
- inputs may be generators;
- do not pre-consume rows to compare lengths;
- reject duplicate header names;
- add row-number context to mismatch errors;
- return a list of dictionaries.

## Solution 7

In [24]:
from collections.abc import Iterable

def records_from_rows(
    headers: Iterable[str],
    rows: Iterable[Iterable[object]],
) -> list[dict[str, object]]:
    header_tuple = tuple(headers)

    if not header_tuple:
        raise ValueError("At least one header is required")

    if len(set(header_tuple)) != len(header_tuple):
        raise ValueError(f"Duplicate headers are not allowed: {header_tuple!r}")

    records: list[dict[str, object]] = []

    for row_number, row in enumerate(rows, start=1):
        try:
            record = dict(zip(header_tuple, row, strict=True))
        except ValueError as exc:
            raise ValueError(
                f"Row {row_number} has a different length from the headers"
            ) from exc
        records.append(record)

    return records

In [25]:
headers = (name for name in ["id", "name", "active"])
rows = (
    row
    for row in [
        [1, "Ada", True],
        [2, "Grace", False],
    ]
)

records = records_from_rows(headers, rows)
assert records == [
    {"id": 1, "name": "Ada", "active": True},
    {"id": 2, "name": "Grace", "active": False},
]

try:
    records_from_rows(["a", "b"], [[1, 2], [3]])
except ValueError as exc:
    assert "Row 2" in str(exc)
else:
    raise AssertionError("Expected row-length mismatch")

### Why not compare lengths first?

Rows may be one-shot iterators. Turning a row into a list solely to inspect its length either consumes it or adds unnecessary buffering. Strict `zip` expresses and enforces the contract in one pass.

# Problem 8 — Strict matrix transpose and dot product

Implement:

1. `transpose_strict(matrix)` that rejects ragged matrices;
2. `dot_strict(left, right)` that rejects unequal vector lengths;
3. `matrix_multiply(left, right)` using both helpers.

Use strict zipping rather than manual length prechecks wherever parallel iteration is the core operation.

## Solution 8

In [26]:
from collections.abc import Iterable, Sequence
from numbers import Real

def transpose_strict(matrix: Iterable[Iterable[Real]]) -> list[list[Real]]:
    rows = [tuple(row) for row in matrix]

    if not rows:
        return []

    try:
        return [list(column) for column in zip(*rows, strict=True)]
    except ValueError as exc:
        raise ValueError("Cannot transpose a ragged matrix") from exc


def dot_strict(left: Iterable[Real], right: Iterable[Real]) -> Real:
    try:
        return sum(a * b for a, b in zip(left, right, strict=True))
    except ValueError as exc:
        raise ValueError("Dot-product vectors must have equal lengths") from exc


def matrix_multiply(
    left: Iterable[Iterable[Real]],
    right: Iterable[Iterable[Real]],
) -> list[list[Real]]:
    left_rows = [tuple(row) for row in left]
    right_rows = [tuple(row) for row in right]

    if not left_rows or not right_rows:
        return []

    right_columns = transpose_strict(right_rows)

    result: list[list[Real]] = []
    for row_index, left_row in enumerate(left_rows):
        output_row = []
        for column_index, right_column in enumerate(right_columns):
            try:
                output_row.append(dot_strict(left_row, right_column))
            except ValueError as exc:
                raise ValueError(
                    "Incompatible matrix dimensions at "
                    f"left row {row_index}, right column {column_index}"
                ) from exc
        result.append(output_row)

    return result

In [27]:
assert transpose_strict([[1, 2, 3], [4, 5, 6]]) == [
    [1, 4],
    [2, 5],
    [3, 6],
]

assert dot_strict([1, 2, 3], [4, 5, 6]) == 32

product = matrix_multiply(
    [[1, 2, 3], [4, 5, 6]],
    [[7, 8], [9, 10], [11, 12]],
)
assert product == [[58, 64], [139, 154]]

try:
    transpose_strict([[1, 2], [3]])
except ValueError as exc:
    assert "ragged" in str(exc)
else:
    raise AssertionError("Expected ragged-matrix rejection")

# Problem 9 — Streaming reconciliation with strict alignment

Two independent generators produce transaction IDs and amounts. Pair them in order and compute a running total.

## Requirements

- detect a source ending early;
- do not materialize the full streams;
- validate each ID and amount;
- return both paired records and the total;
- explain the partial-consumption caveat.

## Solution 9

In [28]:
from decimal import Decimal, InvalidOperation
from collections.abc import Iterable

def reconcile_transactions(
    transaction_ids: Iterable[object],
    raw_amounts: Iterable[object],
) -> tuple[list[tuple[str, Decimal]], Decimal]:
    records: list[tuple[str, Decimal]] = []
    total = Decimal("0")

    try:
        pairs = zip(transaction_ids, raw_amounts, strict=True)

        for position, (transaction_id, raw_amount) in enumerate(pairs, start=1):
            if not isinstance(transaction_id, str) or not transaction_id.strip():
                raise ValueError(f"Invalid transaction ID at position {position}")

            try:
                amount = Decimal(str(raw_amount))
            except (InvalidOperation, ValueError) as exc:
                raise ValueError(
                    f"Invalid amount at position {position}: {raw_amount!r}"
                ) from exc

            if not amount.is_finite():
                raise ValueError(f"Non-finite amount at position {position}")

            clean_id = transaction_id.strip()
            records.append((clean_id, amount))
            total += amount

    except ValueError as exc:
        if "zip()" in str(exc):
            raise ValueError(
                "Transaction ID and amount streams ended at different positions"
            ) from exc
        raise

    return records, total

In [29]:
def id_stream():
    yield "T-001"
    yield "T-002"
    yield "T-003"

def amount_stream():
    yield "10.25"
    yield 5
    yield Decimal("2.75")

records, total = reconcile_transactions(id_stream(), amount_stream())

assert records == [
    ("T-001", Decimal("10.25")),
    ("T-002", Decimal("5")),
    ("T-003", Decimal("2.75")),
]
assert total == Decimal("18.00")

try:
    reconcile_transactions(iter(["T-1", "T-2"]), iter(["1.00"]))
except ValueError as exc:
    assert "different positions" in str(exc)
else:
    raise AssertionError("Expected unequal-stream rejection")

### Partial-consumption caveat

A strict mismatch is discovered only when one iterator ends before another. By that time, earlier elements have already been consumed. For side-effecting sources, design a transaction boundary, checkpoint, or replay strategy rather than assuming the iterators remain untouched after failure.

# Problem 10 — Build a reusable strict mapper

Implement `strict_map(function, *iterables)`.

## Requirements

- behave like `map`, but require all iterables to end together;
- remain lazy;
- support two or more iterables;
- preserve exceptions raised by the supplied function;
- provide a clear error when fewer than two iterables are passed.

## Solution 10

In [30]:
from collections.abc import Callable, Iterable, Iterator
from typing import TypeVar

T = TypeVar("T")

def strict_map(
    function: Callable[..., T],
    *iterables: Iterable[object],
) -> Iterator[T]:
    if len(iterables) < 2:
        raise ValueError("strict_map requires at least two iterables")

    for arguments in zip(*iterables, strict=True):
        yield function(*arguments)

In [31]:
sums = strict_map(
    lambda a, b, c: a + b + c,
    [1, 2, 3],
    [10, 20, 30],
    [100, 200, 300],
)
assert list(sums) == [111, 222, 333]

lazy_result = strict_map(lambda a, b: a * b, iter([2, 3]), iter([4, 5]))
assert iter(lazy_result) is lazy_result
assert next(lazy_result) == 8
assert next(lazy_result) == 15

try:
    next(lazy_result)
except StopIteration:
    pass
else:
    raise AssertionError("Generator should be exhausted")

try:
    list(strict_map(lambda a, b: a + b, [1, 2], [10]))
except ValueError:
    pass
else:
    raise AssertionError("Expected strict length mismatch")

# Problem 11 — Schema-driven row validation with pattern matching and strict zip

Given a schema and a row, validate and convert every field.

A schema entry is one of:

- `("int", nullable)`
- `("float", nullable)`
- `("str", nullable)`
- `("bool", nullable)`

## Requirements

- schema names and row values must have equal lengths;
- dispatch conversion with pattern matching;
- preserve field-level error context;
- support common boolean strings;
- reject unknown schema types.

## Solution 11

In [32]:
from typing import Any

SchemaEntry = tuple[str, bool]

def convert_field(raw: object, schema: SchemaEntry) -> Any:
    subject = (raw, schema)

    match subject:
        case (None, (_, True)):
            return None

        case (None, (_, False)):
            raise ValueError("null is not allowed")

        case (bool(value), ("bool", _)):
            return value

        case (str(text), ("bool", _)) if text.strip().lower() in {"true", "1", "yes"}:
            return True

        case (str(text), ("bool", _)) if text.strip().lower() in {"false", "0", "no"}:
            return False

        case (value, ("int", _)) if not isinstance(value, bool):
            try:
                return int(value)
            except (TypeError, ValueError) as exc:
                raise ValueError(f"cannot convert {value!r} to int") from exc

        case (value, ("float", _)) if not isinstance(value, bool):
            try:
                return float(value)
            except (TypeError, ValueError) as exc:
                raise ValueError(f"cannot convert {value!r} to float") from exc

        case (str(value), ("str", _)):
            return value

        case (value, ("str", _)):
            return str(value)

        case (_, (unknown_type, _)):
            raise ValueError(f"unknown schema type: {unknown_type!r}")


def convert_row(
    field_names: list[str],
    schema: list[SchemaEntry],
    raw_values: list[object],
) -> dict[str, Any]:
    try:
        definitions = zip(field_names, schema, raw_values, strict=True)
        output: dict[str, Any] = {}

        for field_name, field_schema, raw_value in definitions:
            try:
                output[field_name] = convert_field(raw_value, field_schema)
            except ValueError as exc:
                raise ValueError(f"Field {field_name!r}: {exc}") from exc

        return output

    except ValueError as exc:
        if "zip()" in str(exc):
            raise ValueError(
                "Field names, schema entries, and row values must have equal lengths"
            ) from exc
        raise

In [33]:
field_names = ["id", "ratio", "name", "active", "note"]
schema = [
    ("int", False),
    ("float", False),
    ("str", False),
    ("bool", False),
    ("str", True),
]
raw_values = ["7", "0.25", "Ada", "yes", None]

converted = convert_row(field_names, schema, raw_values)

assert converted == {
    "id": 7,
    "ratio": 0.25,
    "name": "Ada",
    "active": True,
    "note": None,
}

try:
    convert_row(["a", "b"], [("int", False)], ["1", "2"])
except ValueError as exc:
    assert "equal lengths" in str(exc)
else:
    raise AssertionError("Expected schema alignment error")

# Problem 12 — Structural log parser

Parse log entries represented in several shapes:

- tuple: `("INFO", timestamp, message)`;
- mapping: `{"level": ..., "timestamp": ..., "message": ..., ...}`;
- exception report: `("ERROR", timestamp, exception_object, context_mapping)`.

Normalize them to a `LogRecord`. Apply guards for timezone awareness and accepted levels.

## Solution 12

In [34]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any

@dataclass(frozen=True)
class LogRecord:
    level: str
    timestamp: datetime
    message: str
    context: dict[str, Any] = field(default_factory=dict)

ACCEPTED_LEVELS = {"DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"}


def parse_log_entry(entry: object) -> LogRecord:
    match entry:
        case (
            "ERROR",
            datetime() as timestamp,
            BaseException() as error,
            dict(context),
        ) if timestamp.tzinfo is not None:
            return LogRecord(
                "ERROR",
                timestamp,
                f"{type(error).__name__}: {error}",
                context,
            )

        case (
            str(level),
            datetime() as timestamp,
            str(message),
        ) if level in ACCEPTED_LEVELS and timestamp.tzinfo is not None:
            return LogRecord(level, timestamp, message, {})

        case {
            "level": str(level),
            "timestamp": datetime() as timestamp,
            "message": str(message),
            **context,
        } if level in ACCEPTED_LEVELS and timestamp.tzinfo is not None:
            return LogRecord(level, timestamp, message, context)

        case (_, datetime() as timestamp, *_) if timestamp.tzinfo is None:
            raise ValueError("Log timestamps must be timezone-aware")

        case _:
            raise ValueError(f"Unsupported log entry: {entry!r}")

In [35]:
from datetime import timezone

now = datetime(2026, 7, 30, 12, 0, tzinfo=timezone.utc)

assert parse_log_entry(("INFO", now, "started")) == LogRecord(
    "INFO", now, "started", {}
)

mapping_record = parse_log_entry(
    {
        "level": "WARNING",
        "timestamp": now,
        "message": "slow request",
        "duration_ms": 900,
    }
)
assert mapping_record.context == {"duration_ms": 900}

error_record = parse_log_entry(
    ("ERROR", now, RuntimeError("boom"), {"job_id": 3})
)
assert error_record.message == "RuntimeError: boom"

# Problem 13 — Capstone: synchronized event-processing pipeline

Combine structural pattern matching and strict zipping.

Three sources provide:

1. event payloads;
2. ingestion timestamps;
3. source names.

They must remain exactly aligned.

## Requirements

- process the streams lazily;
- reject unequal stream lengths;
- normalize supported event payloads with pattern matching;
- attach a timezone-aware ingestion timestamp and non-empty source;
- produce accepted records and rejected records without stopping for ordinary malformed payloads;
- treat stream misalignment as a fatal integrity error;
- include original position in all output.

## Solution 13

In [36]:
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Iterable, Iterator

@dataclass(frozen=True)
class AcceptedRecord:
    position: int
    source: str
    ingested_at: datetime
    event: UserCreated | UserDeleted | InvoicePaid

@dataclass(frozen=True)
class RejectedRecord:
    position: int
    source: str
    ingested_at: datetime
    payload: object
    reason: str


def process_event_streams(
    payloads: Iterable[object],
    timestamps: Iterable[datetime],
    sources: Iterable[str],
) -> Iterator[AcceptedRecord | RejectedRecord]:
    try:
        aligned = zip(payloads, timestamps, sources, strict=True)

        for position, (payload, timestamp, source) in enumerate(aligned, start=1):
            match (timestamp, source):
                case (datetime() as ts, str(src)) if (
                    ts.tzinfo is not None and bool(src.strip())
                ):
                    clean_source = src.strip()
                case _:
                    yield RejectedRecord(
                        position=position,
                        source=str(source),
                        ingested_at=timestamp,
                        payload=payload,
                        reason="Invalid timestamp or source",
                    )
                    continue

            try:
                event = normalize_event(payload)
            except ValueError as exc:
                yield RejectedRecord(
                    position=position,
                    source=clean_source,
                    ingested_at=timestamp,
                    payload=payload,
                    reason=str(exc),
                )
            else:
                yield AcceptedRecord(
                    position=position,
                    source=clean_source,
                    ingested_at=timestamp,
                    event=event,
                )

    except ValueError as exc:
        if "zip()" in str(exc):
            raise RuntimeError(
                "Fatal data-integrity error: event streams are misaligned"
            ) from exc
        raise

In [37]:
from datetime import timezone

payloads = iter([
    {
        "type": "user.created",
        "payload": {
            "id": 1,
            "name": "Ada",
            "email": "ada@example.com",
        },
    },
    {
        "type": "invoice.paid",
        "payload": {
            "invoice_id": 2,
            "currency": "EUR",
            "amount": -5,
        },
    },
    {
        "type": "user.deleted",
        "payload": {"id": 3},
    },
])

timestamps = iter([
    datetime(2026, 7, 30, 10, 0, tzinfo=timezone.utc),
    datetime(2026, 7, 30, 10, 1, tzinfo=timezone.utc),
    datetime(2026, 7, 30, 10, 2, tzinfo=timezone.utc),
])

sources = iter(["web", "billing", "admin"])

results = list(process_event_streams(payloads, timestamps, sources))

assert isinstance(results[0], AcceptedRecord)
assert isinstance(results[1], RejectedRecord)
assert isinstance(results[2], AcceptedRecord)
assert [record.position for record in results] == [1, 2, 3]

results

[AcceptedRecord(position=1, source='web', ingested_at=datetime.datetime(2026, 7, 30, 10, 0, tzinfo=datetime.timezone.utc), event=UserCreated(user_id=1, name='Ada', email='ada@example.com', metadata={})),
 RejectedRecord(position=2, source='billing', ingested_at=datetime.datetime(2026, 7, 30, 10, 1, tzinfo=datetime.timezone.utc), payload={'type': 'invoice.paid', 'payload': {'invoice_id': 2, 'currency': 'EUR', 'amount': -5}}, reason="Malformed or unsupported event type: 'invoice.paid'"),
 AcceptedRecord(position=3, source='admin', ingested_at=datetime.datetime(2026, 7, 30, 10, 2, tzinfo=datetime.timezone.utc), event=UserDeleted(user_id=3, reason=None, metadata={}))]

In [38]:
try:
    list(
        process_event_streams(
            payloads=[{"type": "user.deleted", "payload": {"id": 1}}],
            timestamps=[],
            sources=["admin"],
        )
    )
except RuntimeError as exc:
    assert "misaligned" in str(exc)
else:
    raise AssertionError("Expected fatal alignment failure")

## Capstone review

This pipeline separates two kinds of failure:

- **record-level validation errors** become `RejectedRecord` values, allowing processing to continue;
- **cross-stream alignment errors** are fatal because they destroy the meaning of positional pairing.

That distinction is a useful production design principle: recover from local bad data when possible, but stop on failures that invalidate the entire data contract.

# Additional challenge set with compact solutions

## Challenge A — Match a date command

Accept `["date", year, month, day]`, construct a `date`, and reject impossible calendar dates.

In [39]:
from datetime import date

def parse_date_command(command: object) -> date:
    match command:
        case ["date", int(year), int(month), int(day)] if all(
            type(value) is int for value in (year, month, day)
        ):
            try:
                return date(year, month, day)
            except ValueError as exc:
                raise ValueError(f"Invalid calendar date: {command!r}") from exc
        case _:
            raise ValueError(f"Malformed date command: {command!r}")


assert parse_date_command(["date", 2024, 2, 29]) == date(2024, 2, 29)

## Challenge B — Strict dictionary construction without silent truncation

In [40]:
def strict_dict(keys, values):
    try:
        return dict(zip(keys, values, strict=True))
    except ValueError as exc:
        raise ValueError("Keys and values must contain the same number of items") from exc


assert strict_dict(["a", "b"], [1, 2]) == {"a": 1, "b": 2}

## Challenge C — Parse Cartesian coordinates from several shapes

In [41]:
def parse_coordinate(value: object) -> Point:
    match value:
        case Point() as point:
            return point
        case [int(x) | float(x), int(y) | float(y)]:
            return Point(float(x), float(y))
        case {"x": int(x) | float(x), "y": int(y) | float(y), **extra_fields}:
            return Point(float(x), float(y))
        case str(text):
            parts = [part.strip() for part in text.split(",")]
            match parts:
                case [x_text, y_text]:
                    try:
                        return Point(float(x_text), float(y_text))
                    except ValueError as exc:
                        raise ValueError(f"Invalid coordinate text: {text!r}") from exc
                case _:
                    raise ValueError(f"Invalid coordinate text: {text!r}")
        case _:
            raise ValueError(f"Unsupported coordinate: {value!r}")


assert parse_coordinate([1, 2]) == Point(1.0, 2.0)
assert parse_coordinate({"x": 3, "y": 4, "label": "P"}) == Point(3.0, 4.0)
assert parse_coordinate("5.5, 6.25") == Point(5.5, 6.25)

## Challenge D — Strictly compare expected and actual values

In [42]:
def compare_sequences(expected, actual):
    mismatches = []

    try:
        for index, (expected_item, actual_item) in enumerate(
            zip(expected, actual, strict=True)
        ):
            if expected_item != actual_item:
                mismatches.append((index, expected_item, actual_item))
    except ValueError as exc:
        raise AssertionError("Expected and actual sequences have different lengths") from exc

    return mismatches


assert compare_sequences([1, 2, 3], [1, 9, 3]) == [(1, 2, 9)]

# Final review questions and answers

### 1. Why is `case name:` usually dangerous when you intend to compare a constant?

Because it is a capture pattern. It binds the subject to `name` and usually matches everything. Use a literal, enum member, or other qualified value such as `Color.RED`.

### 2. Why should specific cases precede general cases?

`match` selects the first successful case. A broad earlier pattern can make later, more specific logic unreachable in practice.

### 3. What does a guard do?

A guard adds a Boolean condition to an already successful structural match. The case body runs only when both the pattern and guard succeed.

### 4. Why is `zip(strict=True)` better than comparing iterator lengths first?

Many iterators do not expose a length and are consumed when traversed. Strict zip validates alignment during the single intended traversal.

### 5. Does strict zip guarantee no input was consumed after failure?

No. A mismatch may be detected only after earlier aligned pairs have already been consumed.

### 6. When should ordinary `zip` remain appropriate?

When truncation to the shortest iterable is deliberate and clearly documented.

# Suggested next practice

Modify the capstone so that:

- accepted events are grouped by source;
- invoice totals are accumulated by currency;
- duplicate event identifiers are rejected;
- processing statistics are generated from a strict alignment of metric names and metric values.

A strong solution should keep parsing, validation, domain modeling, and aggregation in separate functions.